<a href="https://colab.research.google.com/github/daniballester-ai/ia-generativa-lab04/blob/main/04_diagnostico_falhas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Diagnóstico de Falhas RAG

Aluna: Danielle Magalhães Ballester   

Diagnóstico dos 3 cenários quebrados: chunk mal dimensionado, retrieval mismatch e alucinação.

Reusa o setup do LAB-002 (client, LLM_MODEL, embed_fn).

## Setup (reuso do LAB-002)

Reusa `client`, `LLM_MODEL`, `embed_fn`, `CORPUS_DIR` do pipeline principal.

**Provider:** Groq (LLM) + all-MiniLM-L6-v2 (embeddings locais).

Cada cenário cria sua própria collection no Chroma com parâmetros quebrados de propósito.

In [ ]:
import importlib.metadata as md
import importlib.util
import os
import subprocess
import sys
from pathlib import Path
from urllib.request import Request, urlopen

PINS = [
    "openai==2.37.0",
    "chromadb==1.5.9",
    "langchain-core==1.4.0",
    "langchain-text-splitters==1.1.2",
    "pypdf",
    "python-dotenv",
]
_ALVO = {
    "openai": "2.37.0",
    "chromadb": "1.5.9",
    "langchain-core": "1.4.0",
    "langchain-text-splitters": "1.1.2",
}

def _precisa_instalar() -> bool:
    for _nome, _ver in _ALVO.items():
        try:
            if md.version(_nome) != _ver:
                return True
        except md.PackageNotFoundError:
            return True
    return False

try:
    _em_colab = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    _em_colab = False

if _precisa_instalar():
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    if _em_colab:
        print("Instalado. Reiniciando o runtime... rode 'Run all' de novo.")
        os.kill(os.getpid(), 9)
    else:
        print("Dependencias instaladas. Reinicie o kernel e rode 'Run all' de novo.")
        import sys as _sys
        _sys.exit(0)
else:
    print("Ambiente OK.")

Ambiente OK.


In [ ]:
def _load_groq_key() -> tuple[str, str]:
    try:
        from google.colab import userdata
        try:
            k = userdata.get("GROQ_API_KEY")
            if k:
                return k, "Colab Secrets"
        except Exception:
            pass
    except ImportError:
        pass
    try:
        from dotenv import find_dotenv, load_dotenv
        dp = find_dotenv(usecwd=True)
        if dp:
            load_dotenv(dp)
    except ImportError:
        pass
    k = os.getenv("GROQ_API_KEY")
    if k:
        return k, ".env local"
    from getpass import getpass
    k = getpass("Cole sua GROQ_API_KEY (https://console.groq.com/keys): ")
    if not k:
        raise RuntimeError("GROQ_API_KEY nao fornecida — abortando.")
    return k, "prompt interativo"

api_key, key_source = _load_groq_key()

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI
from pypdf import PdfReader

client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1",
)
LLM_MODEL = "llama-3.1-8b-instant"
embed_fn = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

CORPUS_DIR = Path("data/corpus")
DIAG_DIR = "data/chroma_diag"

print(f"LLM: {LLM_MODEL} | embeddings: all-MiniLM-L6-v2 (local) | key: {key_source}")

LLM: llama-3.1-8b-instant | embeddings: all-MiniLM-L6-v2 (local) | key: .env local


In [ ]:
# Baixar corpus original se ainda nao existir
_PAPERS = {
    "attention-is-all-you-need.pdf": "https://arxiv.org/pdf/1706.03762v7.pdf",
    "rag-knowledge-intensive-nlp.pdf": "https://arxiv.org/pdf/2005.11401v4.pdf",
    "lost-in-the-middle.pdf": "https://arxiv.org/pdf/2307.03172v3.pdf",
}

if not list(CORPUS_DIR.glob("*.pdf")):
    CORPUS_DIR.mkdir(parents=True, exist_ok=True)
    for _name, _url in _PAPERS.items():
        _req = Request(_url, headers={"User-Agent": "aulas-ppi-corpus/1.0"})
        (CORPUS_DIR / _name).write_bytes(urlopen(_req).read())
        print("baixado:", _name)

pdfs = sorted(CORPUS_DIR.glob("*.pdf"))
print(f"corpus: {len(pdfs)} PDFs em {CORPUS_DIR}/")

corpus: 4 PDFs em data\corpus/


In [ ]:
def ingest_pdfs(corpus_dir: Path) -> list[dict]:
    docs = []
    for pdf_path in sorted(corpus_dir.glob("*.pdf")):
        reader = PdfReader(pdf_path)
        for page_idx, page in enumerate(reader.pages):
            text = page.extract_text() or ""
            if text.strip():
                docs.append({
                    "text": text,
                    "source": pdf_path.name,
                    "page": page_idx + 1,
                })
    return docs

docs = ingest_pdfs(CORPUS_DIR)
print(f"Paginas ingeridas: {len(docs)}")

Paginas ingeridas: 54


---
## Cenário A — Chunk mal dimensionado (200 chars, overlap=0)

**Hipótese:** Chunks de 200 chars sem overlap fragmentam conceitos com explicações longas.
O LLM recebe pedaços incompletos e completa por palpite → **faithfulness cai**.

**Fix:** chunk_size=800, overlap=100 (valores do LAB-002 original).

In [ ]:
splitter_bad = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=0,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks_a = []
for doc in docs:
    for i, chunk in enumerate(splitter_bad.split_text(doc["text"])):
        chunks_a.append({
            "id": f"{doc['source']}-p{doc['page']}-c{i}",
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"],
            "chunk_idx": i,
        })

print(f"Total de chunks (quebrados): {len(chunks_a)}")
print(f"Tamanho medio: {sum(len(c['text']) for c in chunks_a) / len(chunks_a):.0f} chars")

# Indexa no Chroma separado
chroma_diag = chromadb.PersistentClient(path=DIAG_DIR)
try:
    chroma_diag.delete_collection("papers_chunk200")
except Exception:
    pass
coll_bad = chroma_diag.get_or_create_collection(
    name="papers_chunk200",
    embedding_function=embed_fn,
)

BATCH = 50
for start in range(0, len(chunks_a), BATCH):
    lote = chunks_a[start : start + BATCH]
    coll_bad.add(
        ids=[c["id"] for c in lote],
        documents=[c["text"] for c in lote],
        metadatas=[{"source": c["source"], "page": c["page"], "chunk_idx": c["chunk_idx"]} for c in lote],
    )

print(f"Indexados: {coll_bad.count()} chunks")

Total de chunks (quebrados): 1076
Tamanho medio: 166 chars
Indexados: 1076 chunks


In [ ]:
def retrieve_bad(query: str, k: int = 5) -> list[dict]:
    result = coll_bad.query(query_texts=[query], n_results=k)
    return [
        {
            "text": result["documents"][0][i],
            "source": result["metadatas"][0][i]["source"],
            "page": result["metadatas"][0][i]["page"],
            "distance": result["distances"][0][i],
        }
        for i in range(len(result["documents"][0]))
    ]


QUERIES_A = [
    "Qual é a diferença entre fine-tuning e RAG?",
    "O que é RAG e como ele combina retrieval com geração?",
    "Qual o papel de embeddings em busca semântica?",
]

SHOW_PROMPT = """Voce é um assistente técnico. Responda APENAS com base no contexto abaixo.
Se a informação nao estiver no contexto, diga "Não encontrado no corpus".
Sempre cite a fonte usando o formato [arquivo:pagina].

CONTEXTO:
{context}

PERGUNTA: {question}

RESPOSTA:"""

for q in QUERIES_A:
    hits = retrieve_bad(q)
    context = "\n\n---\n\n".join(
        f"[{h['source']}:p{h['page']}]\n{h['text']}" for h in hits
    )
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": SHOW_PROMPT.format(context=context, question=q)}],
        temperature=0.0,
    )
    print(f"\nQ: {q}")
    print(f"A: {response.choices[0].message.content}")

    # Mostra fragmentacao do top-1 chunk
    print(f"\n  >>> Top-1 chunk ({hits[0]['source']}:p{hits[0]['page']}, {len(hits[0]['text'])} chars):")
    print(f"  {hits[0]['text'][:200]}")


Q: Qual é a diferença entre fine-tuning e RAG?
A: Fine-tuning e RAG são duas abordagens distintas para adaptar LLMs a conhecimentos específicos. O fine-tuning é ideal para adaptar estilo, tom e comportamento geral do modelo, enquanto o RAG é ideal para conhecimento factual atualizável frequentemente. [conceitos-fundamentais.pdf:p1, p2]

  >>> Top-1 chunk (conceitos-fundamentais.pdf:p1, 131 chars):
  3. Diferenca entre Fine-tuning e RAG
Fine-tuning e RAG sao duas abordagens distintas para adaptar LLMs a conhecimentos especificos.

Q: O que é RAG e como ele combina retrieval com geração?
A: RAG (Retrieval-Augmented Generation) é um modelo que combina retrieval com geração. Ele funciona da seguinte forma: um sistema externo de recuperação (retriever) busca documentos relevantes em uma base de conhecimento, e o modelo de geração (LLM) gera a resposta condicionada a esses documentos. Isso é feito com o modelo base permanecendo congelado, ou seja, não é treinado novamente.

  >>> Top-1 chun

### Fix Cenário A — chunk_size=800, overlap=100

Re-indexamos o mesmo corpus com chunks maiores e overlap, depois re-rodamos as mesmas 3 queries.

In [ ]:
splitter_fix = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks_a_fix = []
for doc in docs:
    for i, chunk in enumerate(splitter_fix.split_text(doc["text"])):
        chunks_a_fix.append({
            "id": f"{doc['source']}-p{doc['page']}-c{i}",
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"],
        })

try:
    chroma_diag.delete_collection("papers_fix800")
except Exception:
    pass
coll_fix = chroma_diag.get_or_create_collection(name="papers_fix800", embedding_function=embed_fn)

for start in range(0, len(chunks_a_fix), 50):
    lote = chunks_a_fix[start : start + 50]
    coll_fix.add(
        ids=[c["id"] for c in lote],
        documents=[c["text"] for c in lote],
        metadatas=[{"source": c["source"], "page": c["page"]} for c in lote],
    )

print(f"\n--- FIX APLICADO: chunk_size=800, overlap=100 ---")
print(f"Chunks: {len(chunks_a_fix)} (vs {len(chunks_a)} com chunk=200)")

for q in QUERIES_A:
    result = coll_fix.query(query_texts=[q], n_results=5)
    hits_fix = [
        {"text": result["documents"][0][i], "source": result["metadatas"][0][i]["source"], "page": result["metadatas"][0][i]["page"]}
        for i in range(len(result["documents"][0]))
    ]
    ctx = "\n\n---\n\n".join(f"[{h['source']}:p{h['page']}]\n{h['text']}" for h in hits_fix)
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": SHOW_PROMPT.format(context=ctx, question=q)}],
        temperature=0.0,
    )
    print(f"\nQ: {q}")
    print(f"A: {resp.choices[0].message.content}")
    print(f"  >>> Top-1 chunk ({hits_fix[0]['source']}:p{hits_fix[0]['page']}, {len(hits_fix[0]['text'])} chars) — antes tinha ~130 chars")


--- FIX APLICADO: chunk_size=800, overlap=100 ---
Chunks: 279 (vs 1076 com chunk=200)

Q: Qual é a diferença entre fine-tuning e RAG?
A: Fine-tuning e RAG são duas abordagens distintas para adaptar LLMs a conhecimentos específicos. Fine-tuning envolve treinar o modelo base adicionalmente em um dataset específico do domínio, alterando permanentemente os parâmetros do modelo. Já RAG (Retrieval-Augmented Generation) utiliza um modelo base congelado e um sistema externo de recuperação para buscar documentos relevantes em uma base de conhecimento, gerando a resposta condicionada a esses documentos.

[conceitos-fundamentais.pdf:p1]
  >>> Top-1 chunk (conceitos-fundamentais.pdf:p1, 698 chars) — antes tinha ~130 chars

Q: O que é RAG e como ele combina retrieval com geração?
A: RAG (Retrieval-Augmented Generation) é um modelo que combina retrieval com geração. Ele funciona da seguinte forma: um sistema externo de recuperação (retriever) busca documentos relevantes em uma base de conhecimento,

---
## Cenário B — Retrieval Mismatch (vocabulário coloquial vs técnico)

**Hipótese:** O embedding all-MiniLM-L6-v2 não casa queries coloquiais com termos técnicos
do corpus usando apenas similaridade densa → **context_precision cai**.

Usamos a collection "papers_chunk200" (já indexada) para este teste.

**Fix:** reescrever queries com termos técnicos do corpus (query expansion).

In [ ]:
QUERIES_B_COLOQUIAL = [
    "Como o computador aprende sozinho?",
    "Pra que serve aquele negócio de vetor de palavras?",
]

for q in QUERIES_B_COLOQUIAL:
    hits = retrieve_bad(q)
    print(f"\nQ: {q}")
    print(f"  Top-5 recuperados:")
    for h in hits:
        relevante = ""
        if q == "Como o computador aprende sozinho?":
            if "unsupervised" in h['text'].lower() or "self-supervised" in h['text'].lower() or "pre-training" in h['text'].lower():
                relevante = " <<< RELEVANTE"
        elif q == "Pra que serve aquele negócio de vetor de palavras?":
            if "embedding" in h['text'].lower() or "vector" in h['text'].lower() or "dense" in h['text'].lower():
                relevante = " <<< RELEVANTE"
        print(f"    [{h['source']}:p{h['page']}] dist={h['distance']:.3f} | {h['text'][:80]}...{relevante}")


Q: Como o computador aprende sozinho?
  Top-5 recuperados:
    [conceitos-fundamentais.pdf:p1] dist=0.415 | mesmo assunto mesmo quando usam vocabulario diferente. Por exemplo, a query 'Com...
    [conceitos-fundamentais.pdf:p1] dist=0.538 | de informacoes especificas mas correm o risco de perder o contexto necessario pa...
    [conceitos-fundamentais.pdf:p1] dist=0.543 | sozinho?' pode encontrar documentos sobre 'unsupervised learning' porque ambos g... <<< RELEVANTE
    [conceitos-fundamentais.pdf:p1] dist=0.547 | comeca no final de um chunk e termina no inicio do proximo nunca sera recuperado...
    [conceitos-fundamentais.pdf:p1] dist=0.556 | frase ou documento e mapeado para um vetor numerico de modo que textos semantica...

Q: Pra que serve aquele negócio de vetor de palavras?
  Top-5 recuperados:
    [conceitos-fundamentais.pdf:p1] dist=0.468 | frase ou documento e mapeado para um vetor numerico de modo que textos semantica...
    [conceitos-fundamentais.pdf:p1] dist=0.480 | em 

In [ ]:
QUERIES_B_TECNICAS = [
    "O que é unsupervised learning e como funciona o pre-training?",
    "O que são word embeddings e dense vectors em NLP?",
]

RAG_PROMPT_ANCHORED = """Voce é um assistente técnico. Responda APENAS com base no contexto abaixo.
Se a informação não estiver no contexto, diga "Não encontrado no corpus".
Sempre cite a fonte usando o formato [arquivo:pagina].

CONTEXTO:
{context}

PERGUNTA: {question}

RESPOSTA:"""

for q in QUERIES_B_TECNICAS:
    hits = retrieve_bad(q)
    context = "\n\n---\n\n".join(
        f"[{h['source']}:p{h['page']}]\n{h['text']}" for h in hits
    )
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": RAG_PROMPT_ANCHORED.format(context=context, question=q)}],
        temperature=0.0,
    )
    print(f"\nQ: {q}")
    print(f"A: {response.choices[0].message.content}")


Q: O que é unsupervised learning e como funciona o pre-training?
A: O "unsupervised learning" é um tipo de aprendizado de máquina que gera vetores semanticamente próximos, permitindo encontrar documentos relacionados. No entanto, não há informações específicas sobre como funciona o pre-training nesse contexto.

No entanto, o pre-training é mencionado no arquivo [rag-knowledge-intensive-nlp.pdf:p2] como uma técnica que permite acessar conhecimento sem treinamento adicional, utilizando mecanismos de acesso pré-treinados.

Q: O que são word embeddings e dense vectors em NLP?
A: Word embeddings são representações vetoriais densas de texto em um espaço continuo de alta dimensão. Dense vectors são uma forma de representar as palavras como vetores numéricos em um espaço de alta dimensão, permitindo a comparação por similaridade de cosseno em vez de comparação exata de palavras. [conceitos-fundamentais.pdf:p1]


### Fix Cenário B — Query expansion (coloquial → técnico)

Expandimos as queries coloquiais com tradução pt→en via LLM, depois comparamos
quantidade de hits relevantes com e sem expansão.

In [ ]:
def expand_query(q: str) -> str:
    resp = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": (
            "Traduza a pergunta abaixo para ingles tecnico (mantendo o sentido original)."
            " Responda APENAS com a traducao, sem explicacoes.\n\n"
            f"{q}"
        )}],
        temperature=0.0,
    )
    return resp.choices[0].message.content.strip()

print("=== FIX CENARIO B: Query Expansion + Re-medida ===\n")

for colq, tecq in zip(QUERIES_B_COLOQUIAL, QUERIES_B_TECNICAS):
    q_expanded = expand_query(colq)
    print(f"Coloquial: {colq}")
    print(f"  Tecnica:  {tecq}")
    print(f"  Expandida: {q_expanded}")

    # --- sem expansao (coloquial cru) ---
    hits_col = retrieve_bad(colq)
    n_rel_col = sum(
        1 for h in hits_col
        if any(kw in h["text"].lower() for kw in ["aprendizado", "machine learning", "word2vec", "word embedding"])
    )

    # --- com expansao ---
    hits_exp = retrieve_bad(q_expanded)
    n_rel_exp = sum(
        1 for h in hits_exp
        if any(kw in h["text"].lower() for kw in ["aprendizado", "machine learning", "word2vec", "word embedding"])
    )

    print(f"  Hits relevantes: coloquial={n_rel_col}/5, expandida={n_rel_exp}/5")
    print(f"  Melhoria: +{n_rel_exp - n_rel_col} hits\n")

# Re-rodar com query expansion no RAG completo
for colq in QUERIES_B_COLOQUIAL:
    q_exp = expand_query(colq)
    hits = retrieve_bad(q_exp)
    context = "\n\n---\n\n".join(
        f"[{h['source']}:p{h['page']}]\n{h['text']}" for h in hits
    )
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user", "content": RAG_PROMPT_ANCHORED.format(context=context, question=colq)}],
        temperature=0.0,
    )
    print(f"Q (coloquial, com expansão): {colq}")
    print(f"A: {response.choices[0].message.content}\n")

=== FIX CENARIO B: Query Expansion + Re-medida ===

Coloquial: Como o computador aprende sozinho?
  Tecnica:  O que é unsupervised learning e como funciona o pre-training?
  Expandida: How does the computer learn by itself?
  Hits relevantes: coloquial=0/5, expandida=0/5
  Melhoria: +0 hits

Coloquial: Pra que serve aquele negócio de vetor de palavras?
  Tecnica:  O que são word embeddings e dense vectors em NLP?
  Expandida: O que serve para é um vetor de palavras, ou seja, uma representação numérica de um conjunto de palavras, geralmente usada em técnicas de processamento de linguagem natural, como a busca de semelhanças entre textos, a classificação de textos e a recomendação de conteúdo.
  Hits relevantes: coloquial=0/5, expandida=0/5
  Melhoria: +0 hits

Q (coloquial, com expansão): Como o computador aprende sozinho?
A: Não encontrado no corpus.

Q (coloquial, com expansão): Pra que serve aquele negócio de vetor de palavras?
A: Os vetores de palavras servem para representar as pal

---
## Cenário C — Alucinação por prompt sem âncora

**Hipótese:** Sem instrução explícita de ancoragem ao contexto, o LLM completa
a resposta com conhecimento próprio → **faithfulness = 0**.

Query: "Quantos parametros tem o GPT-5?" (informação não existe no corpus).

**Fix:** prompt com âncora e fallback "Não encontrado".

In [ ]:
LOOSE_PROMPT = """Responda a pergunta com base nos textos abaixo.

{context}

PERGUNTA: {question}"""

GPT5_QUERY = "Quantos parametros tem o GPT-5?"

# Usa a collection boa do LAB-002 se existir, ou a collection chunk200 como fallback
try:
    chroma_main = chromadb.PersistentClient(path="data/chroma")
    coll_good = chroma_main.get_collection("papers")
except Exception:
    coll_good = coll_bad

def retrieve_good(query: str, k: int = 5) -> list[dict]:
    if coll_good.count() == 0:
        return []
    result = coll_good.query(query_texts=[query], n_results=min(k, coll_good.count()))
    return [
        {
            "text": result["documents"][0][i],
            "source": result["metadatas"][0][i]["source"],
            "page": result["metadatas"][0][i]["page"],
        }
        for i in range(len(result["documents"][0]))
    ]

hits_c = retrieve_good(GPT5_QUERY)
context_c = "\n\n---\n\n".join(
    f"[{h['source']}:p{h['page']}]\n{h['text']}" for h in hits_c
)

print("=" * 60)
print("PROMPT LOOSE (sem ancora)")
print("=" * 60)
resp_loose = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": LOOSE_PROMPT.format(context=context_c, question=GPT5_QUERY)}],
    temperature=0.0,
)
print(f"Q: {GPT5_QUERY}")
print(f"A: {resp_loose.choices[0].message.content}")

print("\n" + "=" * 60)
print("PROMPT ANCORADO (com fallback)")
print("=" * 60)
resp_anchor = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[{"role": "user", "content": RAG_PROMPT_ANCHORED.format(context=context_c, question=GPT5_QUERY)}],
    temperature=0.0,
)
print(f"Q: {GPT5_QUERY}")
print(f"A: {resp_anchor.choices[0].message.content}")

PROMPT LOOSE (sem ancora)
Q: Quantos parametros tem o GPT-5?
A: Infelizmente, não há informações disponíveis nos textos fornecidos sobre o número de parâmetros do GPT-5. Os textos mencionam o GPT-3.5-Turbo, que tem 770M parâmetros, e o T5-11B, que tem 11 bilhões de parâmetros, mas não há informações sobre o GPT-5.

PROMPT ANCORADO (com fallback)
Q: Quantos parametros tem o GPT-5?
A: Não encontrado no corpus.


---
## Tabela Final de Diagnóstico

Preencha a tabela abaixo com os resultados observados em cada cenário.

| Cenário | Falha Observada | Métrica Afetada | Fix Aplicado | Métrica Depois |
|---|---|---|---|---|
| **A** | Fragmentação de conceitos | faithfulness baixa | chunk_size=800, overlap=100 | faithfulness sobe |
| **B** | Top-5 irrelevantes | context_precision baixa | query rewrite com termos técnicos | context_precision sobe |
| **C** | Resposta inventada | faithfulness = 0 | prompt com âncora e fallback "Não encontrado" | resposta correta de falha |

> **Diagnóstico textual (mínimo 3 linhas por cenário):**
>
> **Cenário A:** (descreva a fragmentação observada nos chunks e como afetou a resposta)
>
> **Cenário B:** (descreva quais hits foram irrelevantes e como a query rewrite melhorou)
>
> **Cenário C:** (descreva a diferença entre a resposta com LOOSE_PROMPT e com prompt ancorado)

## Verificação

- [ ] Os três cenários foram executados e as métricas estão registradas
- [ ] Há diagnóstico textual para cada cenário (pelo menos três linhas por cenário) identificando a causa
- [ ] Um fix por cenário foi aplicado e a métrica foi medida de novo
- [ ] A tabela final está preenchida
